# End-to-end Dog Breed Classification

This notebook builds an end-to-end multi-class image classifier using
TensorFlow 2.x and TensorFlow Hub.

## 1. Problem

Identifying the breed of a dog given an image of a dog.

When I'm sitting at the cafe, and I take a photo of a dog, I want to
know what breed of dog it is.

## 2. Data

The data we're using is from Kaggle's dog breed identification competition.

[https://www.kaggle.com/c/dog-breed-identification/overview](https://www.kaggle.com/c/dog-breed-identification/overview)

## 3. Evaluation

From Kaggle www.kaggle.com/competitions/dog-breed-identification/overview/evaluation

"Submissions are evaluated on a  _Multi Class Log Loss_ between
the predicted probability and the observed target."

Note that the link to the Multi-class Log Loss is **still** broken (as in the video).

The evaluation is a file with prediction probabilities for each dog breed of each test image.

## 4. Features

Some information about the data:

- We're dealing with images (unstructured data) so it's probably
best to use deep learning / transfer learning.
- There are 120 breeds of dogs. (This means that 120 different
classes exist.)
- There are a little over 10k images in the training set. The training
set images have labels.
- There are a little over 10k images in the test set. (The test set
images have no labels because we will predict them.)

Get our workspace ready

In [ ]:
from asyncio import ensure_future

# Import our tools
import cytoolz.curried as ctca
import datetime
import pathlib

In [ ]:
# Import typical data analysis packages
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Import Scikit-Learn packages
import sklearn

In [ ]:
# Import TensorFlow
import tensorflow as tf
import keras
from keras import layers

In [ ]:
print('TF Version: ', tf.__version__)

In [ ]:
# Import TensorFlow Hub for **pre-trained models**
import tensorflow_hub as hub

In [ ]:
print('TF Hub Version: ', hub.__version__)

In [ ]:
# Check for GPU availability
print('GPU', 'available (YESSSS!!!)' if tf.config.list_physical_devices('GPU') else 'NOT AVAILABLE')

## Getting our data ready (turning into Tensors)

Remember, with all machine models, our data **must be** in numerical
format. That's what we'll be performing first: turning our images
into Tensors (numerical representations).

Let's start by accessing our data and checking our labels

In [ ]:
# Checkout the labels of our data
labels_csv = pd.read_csv('./data/labels.csv')
print(labels_csv.describe())
print(labels_csv.head())

In [ ]:
labels_csv.head()

In [ ]:
labels_csv['breed'].value_counts()

In [ ]:
labels_csv['breed'].value_counts().plot.bar(figsize=(20, 10))
plt.show()

In [ ]:
labels_csv['breed'].value_counts().mean()

In [ ]:
# More robust than the `mean()` against outliers
labels_csv['breed'].value_counts().median()

In [ ]:
labels_csv['breed'].value_counts()[:5]

In [ ]:
labels_csv['breed'].value_counts()[-5:]

In [ ]:
# Let's view an image
from IPython.display import Image

In [ ]:
# View a dingo
Image('./data/train/001513dfcb2ffafc82cccf4d8bbaba97.jpg')

#### Getting images and their labels

Let's get a list of all our image file pathnames.

In [ ]:
labels_csv.head()

In [ ]:
# Create path names from image IDs.
path_names = [f'./data/train/{file_name}.jpg' for file_name in labels_csv['id']]

# Check first ten
path_names[:10]

In [ ]:
if (len([pathname for pathname in pathlib.Path('./data/train').iterdir()])) == len(path_names):
    print('Actual filenames equals expected filenames. Proceed!!!')
else:
    print('Actual filenames does not equal expected filenames. Check the directory.')

In [ ]:
# One more check
Image(path_names[9000])

In [ ]:
labels_csv['breed'][9000]

Since we've not got our training image path names in a list, let's
prepare our labels.

In [ ]:
labels = labels_csv['breed']
labels

In [ ]:
# Turn our labels into a `np.array'
labels = np.array(labels)

# An alternative
# labels = labels.csv['breed'].to_numpy()
labels

In [ ]:
len(labels)

In [ ]:
# See if we have **missing data**
len(labels) == len(path_names)

In [ ]:
if len(labels) == len(path_names):
    print('Eureka! Number of labels matches number of path names!')
else:
    print('Number of labels does not match number of path names!')

In [ ]:
# Find the unique label values
unique_breeds = np.unique(labels)
unique_breeds

In [ ]:
len(unique_breeds)

This value is **expected**.

In [ ]:
# Turn a single label into an array of booleans
print(labels[0])
labels[0] == unique_breeds

In [ ]:
# Let's now turn **every** label into a Boolean array
boolean_labels = [label == unique_breeds for label in labels]
boolean_labels[:2]

In [ ]:
# Example: Turning boolean array into **integers**
# Original label
print(labels[0])

# Index of `labels[0]` in `unique_breeds`
print(np.where(unique_breeds == labels[0]))

# Index in array where maximum (`True`) occurs
print(boolean_labels[0].argmax())

# Convert labels to integer
# There should be a 1 where the sample label occurs
print(boolean_labels[0].astype(int))

In [ ]:
print(labels[2])
print(boolean_labels[2].astype(int))

In [ ]:
boolean_labels[:2]

In [ ]:
path_names[:10]

### Creating our own validation set

Since the dataset from Kaggle **does not** come with a validation set,
we're going to create our own.

We'll use `train_test_split()` (but the name is misleading).

In [ ]:
# Split data into features and labels (`X` and `y`)
X = path_names
y = boolean_labels

In [ ]:
len(path_names)

We will start by experimenting with ~ 1k images and increase as needed.

In [ ]:
# Set number of images with which to experiment
NUM_IMAGES = 1000

In [ ]:
# Let's split our data into train and validation sets
from sklearn.model_selection import train_test_split

# Split into training and validation sets
X_train, X_validation, y_train, y_validation = train_test_split(
    X[:NUM_IMAGES],
    y[:NUM_IMAGES],
    test_size=0.2,
    # Perhaps prefer my technique of using
    # `default_rng`.
    random_state=42,
)

In [ ]:
# Be sure to check sizes
len(X_train), len(X_validation), len(y_train), len(y_validation)

Remember,

- It is **far too easy** to create splits of the **wrong size**
- And then **waste time** on issues caused by the wrong sized splits

In [ ]:
# Let's have a "geez" at the training data
X_train[:5], y_train[:2]

### Preprocessing Images (turning images into Tensors)

To preprocess our images into Tensors, we write a function which will:

- Take an image path name as input
- Use TensorFlow to read the file and save it to a variable, for
example, `image`
- Turn our `image` (a jgp) into Tensors
- Resize the `image` to have a shape of (224, 224)
- Return the modified `image`

But... Before we write our function, let's see what importing an
image looks like.

In [ ]:
# Convert an image to a `numpy` `array`

# Import `imread`, a `numpy` function te road an image into a
# `numpy` `array`.
from matplotlib.pyplot import imread
image = imread(path_names[42])
image.shape

# Believe the returned shape is (height, width, color_channel)

In [ ]:
# Get the maximum and minimum values in our image
image.max(), image.min()

In [ ]:
# Have a "look" at our image
image

In [ ]:
tf.constant(image)

In [ ]:
# Do we have the same data?
image[:2] == tf.constant(image)[:2]

Now we've seen what an image looks like as a tensor, let's make a
function to preprocess them.

- Take an image path name as input
- Use TensorFlow to read the file and save it to a variable, for
example, `image`
- Turn our `image` (a jgp) into Tensors
- Normalize our image; that is, convert the color channel values
from the integer range (0, 255) to the floating point range (0, 1)
- Resize the `image` to have a shape of (224, 224)
- Return the modified `image`



In [ ]:
# Examining the steps of our function
tensor = tf.io.read_file(path_names[26])
tensor

In [ ]:
tensor = tf.image.decode_jpeg(tensor, channels=3)
tensor[:2]

In [ ]:
tensor = tf.image.convert_image_dtype(tensor, float)
tensor[:2]

In [ ]:
# Define image size
# The constant is chosen to fit the model we will eventually choose.
IMAGE_SIZE = 224

# Create a function for pro-processing images
def preprocess_image(image_pathname, image_size=IMAGE_SIZE):
    """
    Takes the pathname of an image and converts it to a tensor.

    :param image_pathname: The pathname of the file containing the image
    :param image_size: The size of the image. Default to a square of `IMAGE_SIZE`.
    :return: The tensor representing the image.
    """
    image = tf.io.read_file(image_pathname)

    # Turn the jpeg image into a numerical tensor with 3 color
    # channels: red, green, and blue
    image = tf.image.decode_jpeg(image, channels=3)

    # Convert the color channel value for the integer range (0, 255) to
    # the floating point range (0, 1)
    image = tf.image.convert_image_dtype(image, tf.float32)

    # Resize the image to our desired value (224, 224)
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])

    return image

## Turning our data into batches

Why turn our data into "batches"?

Let's say you're trying to process >~ 10k images in one go. They all
might not fit into memory. The resultant swapping will slow us down
immensely (and may cause a failure).

That's why we process about 32 (the "batch size") images  at a time.
(You can manually adjust the batch size if necessary.)

In order to use TensorFlow effectively, we need our data in the form
of a "tensor tuple) which looks like this: `(image, label)`.

In [ ]:
# Create a function to return a tuple: `(image, label)`
def pair_image_label(image_path, label):
    """
    Takes the pathname of an image and an associated label. Returns
    a tuple of the (tensor) image and the label of the image.
    :param image_path: The pathname of the file containing the image.
    :param label: The label associated with the image.
    :return:A tuple of the (tensor) image and the label of the image.
    """
    image = preprocess_image(image_path)
    return image, label

In [ ]:
# Let's test our function implementation
preprocess_image(X[42]), tf.constant(y [32])

Now we have a function to turn one datum into a `tuple` of `tensors`.
Let's make a function to turn **all** of our data (features (`X`)
and labels (`y`)) into **batches**.

In [ ]:
# Define the batch size. 32 is a good start.
BATCH_SIZE = 32

# Create a function to turn data into batches
def create_data_batches(X, y=None, batch_size=BATCH_SIZE, validation_data=False, test_data=False):
    """
    Create batches of data out of image, `X` and label, `y`, pairs.

    Shuffles the data if it's training data but **does not** shuffle
    the data if it's validation data.

    Also accepts test data as input (no labels).

    :param X: The features as tensors.
    :param y: The labels as tensors.
    :param batch_size: The batch size. Default: BATCH_SIZE (32).
    :param validation_data: `True` if data (`X`) is validation data, `False` otherwise.
    :param test_data: `True` if data (`X`) is test data, `False` otherwise.
    :return: Pairs of (image, label) partitioned into `batch_size` batches.
    """

    # If the data is test dataset, we probably don't have labels
    if test_data:
        print('Creating test data batches...')

        # Only path names - no labels
        data = tf.data.Dataset.from_tensor_slices((tf.constant(X)))
        data_batch = data.map(preprocess_image).batch(batch_size)
        return data_batch

    # If the data is a validation dataset, we **need not shuffle it**.
    elif validation_data:
        print('Creating validation data batches...')

        # Convert path names (  X') **and** labels (`y`
        data = tf.data.Dataset.from_tensor_slices((tf.constant(X),
                                                   tf.constant(y)))

        # Create `(image, label)` tuples
        data = data.map(pair_image_label)

        # Turn the training data into batches
        data_batch = data.batch(batch_size)
        return data_batch

    # Otherwise, must be training batch
    else:
        print('Creating training data batches...')

        # Turn path names and labels into tensors
        data = tf.data.Dataset.from_tensor_slices((tf.constant(X),
                                                   tf.constant(y)))

        # Shuffling the path names and labels **before** image processing
        # is faster than shuffling images. (NOTE: shuffling **path names**
        # much faster than shuffling a `tensor`.)
        data = data.shuffle(buffer_size=len(X))

        # Create `(image, label)` tuples
        data = data.map(pair_image_label)

        # Turn the training data into batches
        data_batch = data.batch(batch_size)
        return data_batch

In [ ]:
# Create training and validation data batches
train_data = create_data_batches(X_train, y_train)
validation_data = create_data_batches(X_validation, y_validation, validation_data=True)

In [ ]:
# Check the different attributes of our data batches
train_data.element_spec, validation_data.element_spec

In [ ]:
# Why 120?
len(y[0])

## Visualizing data batches

Our data is now in batches. However, batches can be hard to understand
or comprehend since a "batch" is typically not part of the problem domain.

To help us understand, let's visualize our batches.

In [ ]:
unique_breeds[y[0].argmax()]

In [ ]:
# Create a function for viewing images in a data batch.

def show_25_images(images, labels):
    """
    Displays a plot of 25 images and their labels from a data batch.

    :param images: The batch of 25 images.
    :param labels: The labels for these images.
    """

    # Setup the figure
    plt.figure(figsize=(10, 10))

    # Loop through the 25 images
    for i in range(25):
        # Create 5 rows x 5 columns subplots
        ax = plt.subplot(5, 5, i + 1) ## row, column, indexA

        # Display an image
        plt.imshow(images[i])

        # Title each image
        plt.title(unique_breeds[labels[i].argmax()])

        # Turn the grid lines off
        plt.axis('off')

In [ ]:
# The data is in a batch of 25 items
train_data

In [ ]:
train_images, train_labels = next(train_data.as_numpy_iterator())
train_images, train_labels

In [ ]:
len(train_images), len(train_labels)

In [ ]:
# Now let's visualize the data in a batch
show_25_images(train_images, train_labels)

In [ ]:
# Now let's visualize the data in other batches (every time we
# run this cell).
train_images, train_labels = next(train_data.as_numpy_iterator())
show_25_images(train_images, train_labels)

## Building a model

Remember that deep learning supports **many** paths to success.
We will apply transfer learning to our problem, but we could apply
other approaches. This saves us:

- Energy
- Cost
- Time (we can get "up and running" more quickly than "starting
from scratch"

And allows us to improve on an existing model

Before we build a model, we must define:

- The **input shape**; that is, the shape of our input images which
are in the form of tensors.
- The **output shape**; that is, our image labels - again in the
form of tensors.
- The URL of the model we wish to use from Kaggle (all TensorFlow Hub
models moved to Kaggle).


In [ ]:
IMAGE_SIZE

In [ ]:
# Setup input shape to the model

# See the `element_spec` of training and validation data that we
# previously calculated. The items in the shape array are
# - Batch (number)
# - Height
# - Width
# - Color channels
INPUT_SHAPE = [None, IMAGE_SIZE, IMAGE_SIZE, 3]

# Setup the output shape of our model
OUTPUT_SHAPE = len(unique_breeds)

# Setup model URL from TensorFLow Hub
MODEL_URL = 'https://www.kaggle.com/models/google/mobilenet-v2/TensorFlow2/130-224-classification/2'

Now that we have our

- Inputs
- Outputs
- Models

ready to go, let's put theme together into a Keras learning model!

Let's create a function that
- Takes the input shape, the output shape and the model we've chose
as **parameters**
- Defines the layers in a Keras model in a sequential fashion
    - Do this first
    - Then this
    - Then that
- Compiles the model (says it should be evaluated and improved)
- Builds the model (tells the model the input shape it will receive)
- Returns the model

All of these steps can be
[found here](https://www.tensorflow.org/guide/keras/overview).

In [ ]:
# Create a function that builds a Keras model
def create_model(input_shape=INPUT_SHAPE, output_shape=OUTPUT_SHAPE, model_url=MODEL_URL):
    print(f'Building model with: {model_url}')

    # Setup the model layers
    model = keras.Sequential([
        # Layer 1 - the input layer
        hub.KerasLayer(
            model_url,
        ),

        # Layer 2 - the output layer
        keras.layers.Dense(units=output_shape, activation='softmax'),
    ])

    # Compile the model
    model.compile(
        loss=keras.losses.CategoricalCrossentropy(),
        optimizer=keras.optimizers.legacy.Adam(),
        metrics=['accuracy'],
    )

    # Build the model
    model.build(input_shape=input_shape)

    # Return the compiled and built model
    return model

In [ ]:
# Create a model using all the defaults
model = None ## removes warning in next cell
try:
    model = create_model()
except ValueError as ve:
    print(f'Error: {ve}')

Hmm... It appears as though the tensorflow API has changed significantly
in the last five years. I will need to investigate the changes to
determine how to change my code to conform. Sigh...

### TensorFlow version issue

The code that Daniel wrote in the next video failed to create a model in my original environment.

I had created a conda environment with the latest version of TensorFlow, 2.18.0. Based on [this issue in Daniel's GitHub repository](https://github.com/mrdbourke/zero-to-mastery-ml/issues/100), I believe the issue is known.

To repair it, as [this issue in Daniel's GitHub repository](https://github.com/mrdbourke/zero-to-mastery-ml/issues/100) describes, I also had to:
- Downgrade my version of Python to 2.10
- Downgrade my version of `numpy` to 1.26
- Change my `conda` environment file, `tf-metal-arm64.yaml`, to reflect these pinned packages
- Rebuild the `conda` environment, `dog-vision`
- Re-open my PyCharm project which recognized the new environment and updated its skeletons (took a bit longer for the `numpy`downgrade that I expected)
- Re-ran all cells in my notebook in PyCharm
- Needed to make a minor correction to the optimizer
	- "WARNING:absl:At this time, the v2.11+ optimizer `tf.keras.optimizers.Adam` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.Adam`."

In [ ]:
# Extract the model summary
model.summary()

In [ ]:
outputs = np.ones(shape=(1, 1, 1280))
outputs

## Creating callbacks

Callbacks are helper functions a model can use during training to do
tasks like:

- Save its progress
- Check its progress
- Stop training early if a model stops improving

We'll create two callbacks:

- One for TensorBoard which helps track the progress of our model
- Another for early stopping which prevents our model from training
too long

### TensorBoard Callback

To setup a TensorBoard callback we need to do three things:

1. Load the TensorBoard notebook extension
2. Create a TensorBoard callback which can save longs to a directory
and pass it to our model's `fit()` function
3. Visual our model's training logs with the `%tensorboard` magic function
    - We'll perform this visualization **after** model training

In [ ]:
# Load TensorBoard notebook extension
#load_ext tensorboard

In [ ]:
import datetime  # allows tagging experiments by time stamp of experiment

# Create a function to build a TensorBoard callback
def create_tensorboard_callback():
    # Create a log directory for storing TensorBoard logs
    # The logs should be stored in a different directory everytime
    # we run an experiment. (The `datetime` component
    log_dir = pathlib.Path(
        '.',
       'logs',
        datetime.datetime.now().isoformat(timespec='seconds')
    )
    return tf.keras.callbacks.TensorBoard(log_dir=log_dir)

### Early stopping callback

Early stopping helps our model from overfitting by stopping training
if a certain evaluation callback **stops** improving.

https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping



In [ ]:
# Create early stopping callback
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=3,
)

## Training a model (on a subset of data)

We will train our first model on only 1000 images. We want to ensure
everything is working before committing large amounts of time.

In [ ]:
# We must define one additional variable before training our model

# Leave value at 100 because we have set our early stopping value
# of 3. (See `patience` above.)
NUM_EPOCHS = 100 # Leave value at 100 because we have

In [ ]:
# One last check to ensure the usage of a GPU
print(
    'GPU',
    ('available (YESSS!!!)' if tf.config.list_physical_devices('GPU')
     else 'not available :(')
)

Let's create a function that trains a model.

- Create a model using `create_model()`
- Setup a TensorBoard callback using `create_tensor_board_callback()`
- Call the `fit()` function on our model passing
    - The training data
    - The validation data
    - The number of epochs for training (`NUM_EPOCHS`)
    - The callbacks we wish to use
- Return the model

In [ ]:
# Build the function to train and return a trained model
def train_model():
    """
    Trains a specified model and returns the trained version.
    :return: The trained model
    """

    # Create a model
    model = create_model()

    # Create a new TensorBoard session for **each** training run.
    tensorboard = create_tensorboard_callback()

    # Fit the model to the data. Pass it the callbacks we have created.
    model.fit(
        x=train_data,
        epochs=NUM_EPOCHS,
        validation_data=validation_data,
        # A value of 1 results in testing once each epoch
        validation_freq=1,
        callbacks=[tensorboard, early_stopping],
    )

    # Return the fitted model
    return model

In [ ]:
model = train_model()

**Question**:

It looks like our model is overfitting because it is performing
far better:

- On the training dataset
- Than on the validation dataset

What are some ways to prevent model overfitting in deep learning neural networks?

**Note**:

Overfitting to begin with is a good thing!

It means that our model is **learning**!!!